# Designing Orchestration Diagrams

Before you wire agents in code, draw the **orchestration diagram**. A good diagram answers: Who runs when? What state crosses each edge? Where are human gates, loops, and failure paths?

Diagrams are not bureaucracy — they are **executable specifications** when treated as diagram-as-code: a structured graph you can render to ASCII or Mermaid, review with stakeholders, and **validate** against a running implementation.

This notebook covers:

1. **Vocabulary** — nodes, edges, gates, and annotation conventions.
2. **Diagram-as-code** — `OrchestrationDiagram` model with renderers.
3. **Pattern library** — sequential, supervisor, nested, group chat, parallel fan-out.
4. **Trace alignment** — compare diagram nodes to runtime step traces.
5. **Design checklist** — what to verify before shipping.

Everything is self-contained plain Python. No frameworks or prior notebooks are required.

In [ ]:
"""
Diagram-as-code toolkit for agentic workflow design.
"""

import json
import os
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


print("Orchestration diagram toolkit ready")

---

## 1. Why Diagram Orchestration First?

| Without a diagram | With a diagram |
|-------------------|----------------|
| "We'll figure out routing in code" | Routing is explicit before LLM calls |
| Hidden loops burn tokens | Loops labeled with `max_retries` |
| Compliance can't audit the system | Diagram is the audit artifact |
| Onboarding takes weeks | New engineers read the graph |

**Rule:** If you cannot draw it, you do not yet understand the workflow.

---

## 2. Diagram Vocabulary

| Symbol / node type | Meaning | Example |
|--------------------|---------|--------|
| `start` / `end` | Workflow boundaries | START, END |
| `agent` | LLM specialist step | `researcher`, `writer` |
| `tool` | Deterministic tool execution | `search_orders` |
| `gate` | Human or rule checkpoint | `human_approve` |
| `router` | Conditional branch | `if approved → publish` |
| `subflow` | Nested workflow black box | `ResearchSubflow` |
| `parallel` | Fan-out / join | legal ∥ security → merge |

**Edge labels** carry hand-off meaning: `artifacts.draft`, `approved=true`, `on_error`.

---

## 3. OrchestrationDiagram Model

Store diagrams as data — render to multiple formats from one source of truth.

In [ ]:
class NodeKind(str, Enum):
    START = "start"
    END = "end"
    AGENT = "agent"
    TOOL = "tool"
    GATE = "gate"
    ROUTER = "router"
    SUBFLOW = "subflow"
    PARALLEL = "parallel"


@dataclass
class DiagramNode:
    id: str
    label: str
    kind: NodeKind
    reads: list[str] = field(default_factory=list)
    writes: list[str] = field(default_factory=list)
    notes: str = ""


@dataclass
class DiagramEdge:
    source: str
    target: str
    label: str = ""
    condition: str = ""


@dataclass
class OrchestrationDiagram:
    name: str
    version: str
    description: str = ""
    nodes: list[DiagramNode] = field(default_factory=list)
    edges: list[DiagramEdge] = field(default_factory=list)

    def add_node(self, node: DiagramNode) -> None:
        self.nodes.append(node)

    def add_edge(self, source: str, target: str, label: str = "", condition: str = "") -> None:
        self.edges.append(DiagramEdge(source, target, label, condition))

    def node_ids(self) -> set[str]:
        return {n.id for n in self.nodes}

    def agent_nodes(self) -> list[DiagramNode]:
        return [n for n in self.nodes if n.kind in (NodeKind.AGENT, NodeKind.SUBFLOW)]


sample = OrchestrationDiagram("demo", "0.1.0", description="Minimal sample")
sample.add_node(DiagramNode("start", "START", NodeKind.START))
sample.add_node(DiagramNode("writer", "Writer", NodeKind.AGENT, writes=["draft"]))
sample.add_node(DiagramNode("end", "END", NodeKind.END))
sample.add_edge("start", "writer")
sample.add_edge("writer", "end", label="draft")
print(f"Sample diagram: {len(sample.nodes)} nodes, {len(sample.edges)} edges")

---

## 4. ASCII Renderer

Quick terminal-friendly output for notebooks, logs, and code reviews.

In [ ]:
def render_ascii(diagram: OrchestrationDiagram) -> str:
    """Linearize diagram edges into an ASCII chain (best for sequential graphs)."""
    lines = [
        f"=== {diagram.name} v{diagram.version} ===",
        diagram.description,
        "",
    ]
    # Build adjacency for simple path walk from start
    adj: dict[str, list[DiagramEdge]] = {}
    for e in diagram.edges:
        adj.setdefault(e.source, []).append(e)
    labels = {n.id: n.label for n in diagram.nodes}
    kinds = {n.id: n.kind.value for n in diagram.nodes}

    start_ids = [n.id for n in diagram.nodes if n.kind == NodeKind.START]
    current = start_ids[0] if start_ids else diagram.nodes[0].id
    visited: set[str] = set()
    path_parts: list[str] = []

    while current and current not in visited:
        visited.add(current)
        path_parts.append(f"[{kinds.get(current, '?')}] {labels.get(current, current)}")
        outs = adj.get(current, [])
        if not outs:
            break
        edge = outs[0]
        if edge.label or edge.condition:
            path_parts.append(f"--({edge.label or edge.condition})-->")
        else:
            path_parts.append("-->")
        current = edge.target

    lines.append(" ".join(path_parts))
    lines.append("")
    lines.append("Nodes:")
    for n in diagram.nodes:
        rw = f" reads={n.reads}" if n.reads else ""
        ww = f" writes={n.writes}" if n.writes else ""
        lines.append(f"  - {n.id} ({n.kind.value}): {n.label}{rw}{ww}")
    return "\n".join(lines)


print(render_ascii(sample))

---

## 5. Mermaid Renderer

Mermaid renders in GitHub, many wikis, and documentation sites — ideal for design reviews.

In [ ]:
MERMAID_SHAPES = {
    NodeKind.START: ("([", "])"),
    NodeKind.END: ("([", "])"),
    NodeKind.AGENT: ("[", "]"),
    NodeKind.TOOL: ("[[", "]]"),
    NodeKind.GATE: ("{{", "}}"),
    NodeKind.ROUTER: ("{", "}"),
    NodeKind.SUBFLOW: ("[", "]"),
    NodeKind.PARALLEL: ("[", "]"),
}


def render_mermaid(diagram: OrchestrationDiagram, direction: str = "TD") -> str:
    lines = [f"---", f"title: {diagram.name} v{diagram.version}", f"---", f"flowchart {direction}"]
    for n in diagram.nodes:
        left, right = MERMAID_SHAPES.get(n.kind, ("[", "]"))
        safe_label = n.label.replace('"', "'")
        lines.append(f"    {n.id}{left}{safe_label}{right}")
    for e in diagram.edges:
        lbl = e.label or e.condition
        if lbl:
            safe = lbl.replace('"', "'")
            lines.append(f"    {e.source} -->|{safe}| {e.target}")
        else:
            lines.append(f"    {e.source} --> {e.target}")
    return "\n".join(lines)


print(render_mermaid(sample))

---

## 6. Pattern — Sequential ReleaseFlow

Diagram for a linear release pipeline with a review loop.

In [ ]:
def build_sequential_release_diagram() -> OrchestrationDiagram:
    d = OrchestrationDiagram(
        name="ReleaseFlow-Sequential",
        version="1.0.0",
        description="Research → write ↔ review → publish",
    )
    nodes = [
        ("start", "START", NodeKind.START, [], []),
        ("researcher", "Researcher", NodeKind.AGENT, ["goal"], ["research"]),
        ("writer", "Writer", NodeKind.AGENT, ["research", "review_feedback"], ["draft"]),
        ("critic", "Critic", NodeKind.AGENT, ["draft"], ["approved", "review_feedback"]),
        ("router", "Approved?", NodeKind.ROUTER, ["approved"], []),
        ("publisher", "Publisher", NodeKind.GATE, ["draft", "approved"], ["announcement_id"]),
        ("end", "END", NodeKind.END, [], []),
    ]
    for nid, label, kind, reads, writes in nodes:
        d.add_node(DiagramNode(nid, label, kind, reads=reads, writes=writes))
    d.add_edge("start", "researcher")
    d.add_edge("researcher", "writer", label="research")
    d.add_edge("writer", "critic", label="draft")
    d.add_edge("critic", "router", label="verdict")
    d.add_edge("router", "writer", condition="rejected & retries<max")
    d.add_edge("router", "publisher", condition="approved")
    d.add_edge("publisher", "end", label="published")
    return d


seq_diagram = build_sequential_release_diagram()
print(render_ascii(seq_diagram))
print("\n--- Mermaid ---\n")
print(render_mermaid(seq_diagram))

---

## 7. Pattern — Supervisor (ShopCo Support)

Router node delegates to workers until done.

In [ ]:
def build_supervisor_diagram() -> OrchestrationDiagram:
    d = OrchestrationDiagram(
        name="ShopCo-Supervisor",
        version="1.0.0",
        description="Supervisor routes to order/policy/billing workers",
    )
    for nid, label, kind in [
        ("start", "START", NodeKind.START),
        ("supervisor", "Top Supervisor", NodeKind.ROUTER),
        ("orders", "Orders Worker", NodeKind.AGENT),
        ("policy", "Policy Worker", NodeKind.AGENT),
        ("billing", "Billing Worker", NodeKind.AGENT),
        ("end", "END", NodeKind.END),
    ]:
        d.add_node(DiagramNode(nid, label, kind))
    d.add_edge("start", "supervisor", label="user_query")
    d.add_edge("supervisor", "orders", condition="intent=order_status")
    d.add_edge("supervisor", "policy", condition="intent=return_policy")
    d.add_edge("supervisor", "billing", condition="intent=billing")
    d.add_edge("orders", "supervisor", label="observation")
    d.add_edge("policy", "supervisor", label="observation")
    d.add_edge("billing", "supervisor", label="observation")
    d.add_edge("supervisor", "end", condition="resolved")
    return d


sup_diagram = build_supervisor_diagram()
print(render_mermaid(sup_diagram))

---

## 8. Pattern — Nested Sub-flows

In [ ]:
def build_nested_diagram() -> OrchestrationDiagram:
    d = OrchestrationDiagram(
        name="ReleaseFlow-Nested",
        version="1.0.0",
        description="Outer pipeline with research and content sub-flows",
    )
    for nid, label, kind in [
        ("start", "START", NodeKind.START),
        ("triage", "Triage", NodeKind.TOOL),
        ("research_sf", "ResearchSubflow", NodeKind.SUBFLOW),
        ("content_sf", "ContentSubflow", NodeKind.SUBFLOW),
        ("publish", "Publisher", NodeKind.GATE),
        ("end", "END", NodeKind.END),
    ]:
        d.add_node(DiagramNode(nid, label, kind))
    d.add_edge("start", "triage")
    d.add_edge("triage", "research_sf", label="goal")
    d.add_edge("research_sf", "content_sf", label="research_bundle")
    d.add_edge("content_sf", "publish", label="formatted_draft, approved")
    d.add_edge("publish", "end")
    return d


print(render_ascii(build_nested_diagram()))

---

## 9. Pattern — Group Chat with Manager

In [ ]:
def build_group_chat_diagram() -> OrchestrationDiagram:
    d = OrchestrationDiagram(
        name="ReleaseFlow-GroupChat",
        version="1.0.0",
        description="Shared transcript; manager selects next speaker",
    )
    d.add_node(DiagramNode("start", "START", NodeKind.START))
    d.add_node(DiagramNode("transcript", "Shared Transcript", NodeKind.TOOL, notes="messages[]"))
    d.add_node(DiagramNode("manager", "GroupChatManager", NodeKind.ROUTER))
    for agent in ("researcher", "writer", "critic"):
        d.add_node(DiagramNode(agent, agent.title(), NodeKind.AGENT))
    d.add_node(DiagramNode("end", "END", NodeKind.END))
    d.add_edge("start", "transcript", label="user goal")
    d.add_edge("transcript", "manager")
    for agent in ("researcher", "writer", "critic"):
        d.add_edge("manager", agent, condition="selected")
        d.add_edge(agent, "transcript", label="reply")
    d.add_edge("manager", "end", condition="TERMINATE or max_rounds")
    return d


print(render_mermaid(build_group_chat_diagram()))

---

## 10. Pattern — Parallel Fan-out / Join

Independent review branches run concurrently, then merge.

In [ ]:
def build_parallel_diagram() -> OrchestrationDiagram:
    d = OrchestrationDiagram(
        name="ReleaseFlow-ParallelReview",
        version="1.0.0",
        description="Legal and security review in parallel before publish",
    )
    for nid, label, kind in [
        ("start", "START", NodeKind.START),
        ("draft_ready", "Draft Ready", NodeKind.TOOL),
        ("fork", "Parallel Fork", NodeKind.PARALLEL),
        ("legal", "Legal Review", NodeKind.AGENT),
        ("security", "Security Review", NodeKind.AGENT),
        ("join", "Merge Reviews", NodeKind.ROUTER),
        ("publish", "Publisher", NodeKind.GATE),
        ("end", "END", NodeKind.END),
    ]:
        d.add_node(DiagramNode(nid, label, kind))
    d.add_edge("start", "draft_ready")
    d.add_edge("draft_ready", "fork")
    d.add_edge("fork", "legal", label="branch A")
    d.add_edge("fork", "security", label="branch B")
    d.add_edge("legal", "join", label="legal_ok")
    d.add_edge("security", "join", label="sec_ok")
    d.add_edge("join", "publish", condition="both approved")
    d.add_edge("publish", "end")
    return d


print(render_mermaid(build_parallel_diagram()))

---

## 11. Annotating State on the Diagram

Put **reads/writes** on nodes (as in section 6) so the diagram doubles as a **contract document**.

In [ ]:
def contract_table(diagram: OrchestrationDiagram) -> str:
    lines = [f"{'Node':<16} {'Kind':<10} {'Reads':<28} {'Writes'}"]
    lines.append("-" * 80)
    for n in diagram.nodes:
        if n.kind in (NodeKind.START, NodeKind.END):
            continue
        lines.append(
            f"{n.id:<16} {n.kind.value:<10} {str(n.reads):<28} {n.writes}"
        )
    return "\n".join(lines)


print(contract_table(seq_diagram))

---

## 12. Diagram Validation — Structural Checks

Catch broken graphs before implementation.

In [ ]:
@dataclass
class ValidationIssue:
    severity: str  # error | warning
    message: str


def validate_diagram(diagram: OrchestrationDiagram) -> list[ValidationIssue]:
    issues: list[ValidationIssue] = []
    ids = diagram.node_ids()

    starts = [n for n in diagram.nodes if n.kind == NodeKind.START]
    ends = [n for n in diagram.nodes if n.kind == NodeKind.END]
    if len(starts) != 1:
        issues.append(ValidationIssue("error", f"expected 1 START node, found {len(starts)}"))
    if len(ends) < 1:
        issues.append(ValidationIssue("error", "no END node"))

    for e in diagram.edges:
        if e.source not in ids:
            issues.append(ValidationIssue("error", f"edge source missing: {e.source}"))
        if e.target not in ids:
            issues.append(ValidationIssue("error", f"edge target missing: {e.target}"))

    # Orphan nodes (no in/out edges) except START/END
    connected = set()
    for e in diagram.edges:
        connected.add(e.source)
        connected.add(e.target)
    for n in diagram.nodes:
        if n.kind not in (NodeKind.START, NodeKind.END) and n.id not in connected:
            issues.append(ValidationIssue("warning", f"orphan node: {n.id}"))

    # Agent nodes should declare writes
    for n in diagram.agent_nodes():
        if not n.writes and n.kind == NodeKind.AGENT:
            issues.append(ValidationIssue("warning", f"agent {n.id} has no writes documented"))

    return issues


for label, diag in [
    ("sequential", seq_diagram),
    ("supervisor", sup_diagram),
    ("nested", build_nested_diagram()),
]:
    issues = validate_diagram(diag)
    status = "PASS" if not any(i.severity == "error" for i in issues) else "FAIL"
    print(f"{label}: {status} ({len(issues)} issues)")
    for i in issues:
        print(f"  [{i.severity}] {i.message}")

---

## 13. Align Diagram with Runtime Trace

After a workflow runs, compare **expected agent nodes** from the diagram to **actual steps** in the trace.

In [ ]:
# Minimal sequential runner whose trace we validate against seq_diagram
CHANGELOG = [
    {"version": "2.4.0", "item": "Added bulk export endpoint"},
    {"version": "2.4.0", "item": "Deprecated legacy webhook"},
]


def run_sequential_trace(goal: str) -> list[str]:
    trace = ["researcher", "writer", "critic"]
    if "retry" in goal.lower():
        trace.extend(["writer", "critic"])
    trace.append("publisher")
    return trace


def align_trace_to_diagram(diagram: OrchestrationDiagram, runtime_steps: list[str]) -> dict[str, Any]:
    expected_agents = [n.id for n in diagram.agent_nodes() if n.kind == NodeKind.AGENT]
    expected_set = set(expected_agents)
    runtime_set = set(runtime_steps)
    return {
        "expected_agents": expected_agents,
        "runtime_steps": runtime_steps,
        "unexpected_runtime": sorted(runtime_set - expected_set),
        "never_ran": sorted(expected_set - runtime_set),
        "ordered_match": all(s in expected_set for s in runtime_steps),
    }


happy_trace = run_sequential_trace("v2.4.0 announcement")
retry_trace = run_sequential_trace("v2.4.0 with retry")

print("Happy path alignment:")
print(pretty(align_trace_to_diagram(seq_diagram, happy_trace)))
print("\nRetry path alignment:")
print(pretty(align_trace_to_diagram(seq_diagram, retry_trace)))

---

## 14. Export Diagram JSON — Version Control

Store diagrams in git alongside code; bump `version` when the graph changes.

In [ ]:
def diagram_to_dict(diagram: OrchestrationDiagram) -> dict[str, Any]:
    return {
        "name": diagram.name,
        "version": diagram.version,
        "description": diagram.description,
        "nodes": [
            {"id": n.id, "label": n.label, "kind": n.kind.value,
             "reads": n.reads, "writes": n.writes, "notes": n.notes}
            for n in diagram.nodes
        ],
        "edges": [
            {"source": e.source, "target": e.target, "label": e.label, "condition": e.condition}
            for e in diagram.edges
        ],
        "exported_at": datetime.now(timezone.utc).isoformat(),
    }


def diagram_from_dict(data: dict[str, Any]) -> OrchestrationDiagram:
    d = OrchestrationDiagram(data["name"], data["version"], data.get("description", ""))
    for n in data["nodes"]:
        d.add_node(DiagramNode(
            n["id"], n["label"], NodeKind(n["kind"]),
            reads=n.get("reads", []), writes=n.get("writes", []), notes=n.get("notes", ""),
        ))
    for e in data["edges"]:
        d.add_edge(e["source"], e["target"], e.get("label", ""), e.get("condition", ""))
    return d


exported = diagram_to_dict(seq_diagram)
roundtrip = diagram_from_dict(exported)
print(f"Round-trip: {roundtrip.name} v{roundtrip.version} — {len(roundtrip.nodes)} nodes")
print("JSON snippet:")
print(pretty({"name": exported["name"], "version": exported["version"], "node_count": len(exported["nodes"])}))

---

## 15. Design Review Checklist

Before implementation sign-off:

1. **Single START, ≥1 END** — graph has clear entry and exit.
2. **Every agent documents reads/writes** — contract table complete.
3. **Loops labeled** — `max_retries`, `max_rounds` on back-edges.
4. **Gates before side effects** — publish, email, charge behind approval nodes.
5. **Failure edges** — what happens on tool error or rejection?
6. **Parallel joins defined** — fan-out has explicit merge condition.
7. **Diagram version** matches code / config version.
8. **Trace alignment test** — sample run matches expected agent sequence.

In [ ]:
DESIGN_CHECKLIST = [
    "single START and at least one END",
    "agent nodes have reads/writes documented",
    "loops show max_retries or max_rounds",
    "side-effect nodes behind gates",
    "error/reject paths drawn",
    "parallel fan-out has join condition",
    "diagram version bumped with code change",
    "sample trace aligns with diagram agents",
]


def checklist_report(diagram: OrchestrationDiagram, runtime_steps: list[str] | None = None) -> dict[str, bool]:
    validation = validate_diagram(diagram)
    has_start_end = not any("START" in i.message or "END" in i.message for i in validation if i.severity == "error")
    agents_documented = all(n.writes for n in diagram.agent_nodes() if n.kind == NodeKind.AGENT)
    has_loop_label = any(e.condition and "retry" in e.condition.lower() for e in diagram.edges)
    has_gate = any(n.kind == NodeKind.GATE for n in diagram.nodes)
    align_ok = True
    if runtime_steps:
        align = align_trace_to_diagram(diagram, runtime_steps)
        align_ok = align["ordered_match"] and not align["unexpected_runtime"]
    return {
        "structure_valid": has_start_end,
        "agents_documented": agents_documented,
        "loop_labeled": has_loop_label,
        "has_gate": has_gate,
        "trace_aligned": align_ok,
    }


report = checklist_report(seq_diagram, happy_trace)
print(f"Design review — {seq_diagram.name} v{seq_diagram.version}")
for i, item in enumerate(DESIGN_CHECKLIST, 1):
    key = list(report.keys())[min(i - 1, len(report) - 1)] if i <= len(report) else None
    mark = "?"
    if i == 1: mark = "✓" if report["structure_valid"] else "✗"
    elif i == 2: mark = "✓" if report["agents_documented"] else "✗"
    elif i == 3: mark = "✓" if report["loop_labeled"] else "○"
    elif i == 4: mark = "✓" if report["has_gate"] else "○"
    elif i == 8: mark = "✓" if report["trace_aligned"] else "✗"
    else: mark = "○"
    print(f"  {i:2}. [{mark}] {item}")

---

## 16. Hands-on — Build Your Own Diagram

In [ ]:
def build_custom_diagram() -> OrchestrationDiagram:
    """Template: customize for your product workflow."""
    d = OrchestrationDiagram(
        name="MyProduct-Workflow",
        version="0.1.0",
        description="Intake → specialist → human gate → action",
    )
    d.add_node(DiagramNode("start", "START", NodeKind.START))
    d.add_node(DiagramNode("intake", "Intake Classifier", NodeKind.TOOL, writes=["intent", "entities"]))
    d.add_node(DiagramNode("specialist", "Domain Agent", NodeKind.AGENT, reads=["intent"], writes=["result"]))
    d.add_node(DiagramNode("human", "Human Approve", NodeKind.GATE, reads=["result"]))
    d.add_node(DiagramNode("action", "Execute Action", NodeKind.TOOL, reads=["result"]))
    d.add_node(DiagramNode("end", "END", NodeKind.END))
    d.add_edge("start", "intake")
    d.add_edge("intake", "specialist", label="intent")
    d.add_edge("specialist", "human", label="result")
    d.add_edge("human", "action", condition="approved")
    d.add_edge("action", "end")
    return d


custom = build_custom_diagram()
print(render_mermaid(custom))
print("\n" + contract_table(custom))

---

## 17. From Diagram to Implementation

| Diagram element | Code mapping |
|-----------------|--------------|
| `agent` node | Agent class `run(state)` or graph node function |
| `tool` node | Deterministic function, no LLM |
| `gate` node | `if approved:` block or HITL pause |
| `router` node | `conditional_edges` or `if/else` router |
| `subflow` node | Nested orchestrator class |
| Edge label | Handoff packet key in shared state |
| `parallel` node | `asyncio.gather` or task queue branches |

**Workflow:** diagram review → contract table → implement nodes one-to-one → trace alignment test.

---

## 18. Quiz

1. What are the minimum structural requirements for a valid orchestration diagram?
2. Why document reads/writes on diagram nodes?
3. How does a `subflow` node differ from an `agent` node on a diagram?
4. What is trace alignment and why run it?
5. When would you use parallel fan-out on a diagram?

---

## 19. Summary

| Concept | Takeaway |
|---------|----------|
| Diagram-first | Draw before coding; expose routing and gates |
| Diagram-as-code | `OrchestrationDiagram` → ASCII / Mermaid / JSON |
| Node kinds | agent, tool, gate, router, subflow, parallel |
| Contracts | reads/writes on nodes = hand-off documentation |
| Validation | Structural checks + trace alignment |
| Pattern library | Sequential, supervisor, nested, group chat, parallel |
| Versioning | Bump diagram `version` with workflow changes |

Treat orchestration diagrams as **living specs**: update them when the graph changes, and fail CI when runtime traces drift from the documented agent sequence.